# NYC Rolling Sales — Statistical Analysis

**Objective:** Perform correlation analysis, regression modeling, and hypothesis testing to uncover statistical relationships in NYC real estate sales data.

**Sections:**
1. Data Loading & Preparation
2. Descriptive Statistics & Distributions
3. Correlation Analysis (Pearson & Spearman)
4. Statistical Hypothesis Testing
5. Regression Analysis
6. Key Findings & Summary

## 1. Data Loading & Setup

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_ind, f_oneway, chi2_contingency
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# ── LOAD CLEANED DATA ────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f"Data shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# ── IDENTIFY NUMERIC & CATEGORICAL COLUMNS ──────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}):")
print(numeric_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):")
print(categorical_cols)

In [ ]:
# ── PREPARE DATA FOR ANALYSIS ───────────────────────────────────────────
# Create a working copy with rows where key variables are not null
analysis_df = df.dropna(subset=['sale_price', 'gross_square_feet']).copy()

print(f"Original shape: {df.shape}")
print(f"Analysis dataset shape: {analysis_df.shape}")
print(f"Records removed: {df.shape[0] - analysis_df.shape[0]}")
print(f"\nMissing values in analysis dataset:")
print(analysis_df[numeric_cols].isnull().sum())

## 2. Descriptive Statistics & Distributions

In [ ]:
# ── SUMMARY STATISTICS FOR KEY VARIABLES ────────────────────────────────
key_vars = ['sale_price', 'price_per_sqft', 'price_per_unit', 'gross_square_feet',
            'land_square_feet', 'residential_units', 'commercial_units', 'building_age']
key_vars = [col for col in key_vars if col in numeric_cols]

summary_stats = analysis_df[key_vars].describe().T
summary_stats['skew'] = analysis_df[key_vars].skew()
summary_stats['kurtosis'] = analysis_df[key_vars].kurtosis()

print("Summary Statistics (Mean, Std, Min, Max, Skew, Kurtosis):")
print(summary_stats)

In [ ]:
# ── VISUALIZE DISTRIBUTIONS OF KEY VARIABLES ───────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(key_vars[:6]):
    axes[idx].hist(analysis_df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../notebooks/stat_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Distribution plots saved.")

## 3. Correlation Analysis

In [ ]:
# ── PEARSON CORRELATION MATRIX ──────────────────────────────────────────
# For continuous variables
correlation_vars = [col for col in numeric_cols if col in analysis_df.columns and col not in ['borough', 'sale_year', 'sale_month', 'sale_quarter']]
corr_matrix = analysis_df[correlation_vars].corr(method='pearson')

print("Pearson Correlation Matrix:")
print(corr_matrix)
print(f"\nCorrelation matrix shape: {corr_matrix.shape}")

In [ ]:
# ── VISUALIZE CORRELATION HEATMAP ───────────────────────────────────────
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Pearson Correlation Matrix - NYC Rolling Sales')
plt.tight_layout()
plt.savefig('../notebooks/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("Correlation heatmap saved.")

In [ ]:
# ── EXTRACT TOP CORRELATIONS WITH SALE_PRICE ────────────────────────────
if 'sale_price' in corr_matrix.index:
    price_corr = corr_matrix['sale_price'].sort_values(ascending=False)
    print("\nTop Correlations with Sale Price:")
    print(price_corr[1:11])  # Exclude self-correlation
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    price_corr[1:11].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 10 Correlations with Sale Price')
    ax.set_xlabel('Pearson Correlation Coefficient')
    plt.tight_layout()
    plt.savefig('../notebooks/price_correlations.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# ── SPEARMAN CORRELATION (FOR RANK-ORDER RELATIONSHIPS) ─────────────────
spearman_corr = analysis_df[correlation_vars].corr(method='spearman')

if 'sale_price' in spearman_corr.index:
    spearman_price = spearman_corr['sale_price'].sort_values(ascending=False)
    print("\nTop Spearman Correlations with Sale Price:")
    print(spearman_price[1:11])

## 4. Statistical Hypothesis Testing

In [ ]:
# ── TEST 1: NORMALITY TESTS (SHAPIRO-WILK) FOR KEY VARIABLES ────────────
print("="*80)
print("NORMALITY TESTS (Shapiro-Wilk)")
print("="*80)
print("Null Hypothesis: Data is normally distributed")
print("\nResults (sample of up to 5,000 rows for computational efficiency):")

for col in key_vars[:5]:
    sample = analysis_df[col].dropna().sample(min(5000, len(analysis_df[col].dropna())), random_state=42)
    stat, p_value = stats.shapiro(sample)
    is_normal = "YES" if p_value > 0.05 else "NO"
    print(f"\n{col}:")
    print(f"  Test Statistic: {stat:.6f}")
    print(f"  p-value: {p_value:.6e}")
    print(f"  Normal? {is_normal} (α=0.05)")

In [ ]:
# ── TEST 2: ANOVA - DIFFERENCES IN PRICE ACROSS BOROUGHS ─────────────────
print("\n" + "="*80)
print("ANOVA TEST - Sale Price across Boroughs")
print("="*80)
print("Null Hypothesis: Mean sale price is equal across all boroughs")

# Group prices by borough
boroughs = analysis_df['borough_name'].unique()
borough_prices = [analysis_df[analysis_df['borough_name'] == boro]['sale_price'].dropna() for boro in boroughs]

# ANOVA test
f_stat, p_value = f_oneway(*borough_prices)
print(f"\nF-Statistic: {f_stat:.6f}")
print(f"p-value: {p_value:.6e}")
print(f"\nConclusion: {'Reject null hypothesis' if p_value < 0.05 else 'Fail to reject null hypothesis'} (α=0.05)")

# Descriptive stats by borough
print("\nMean Sale Price by Borough:")
borough_price_stats = analysis_df.groupby('borough_name')['sale_price'].agg(['mean', 'median', 'std', 'count'])
print(borough_price_stats)

In [ ]:
# ── VISUALIZE PRICE DISTRIBUTION BY BOROUGH ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
analysis_df.boxplot(column='sale_price', by='borough_name', ax=ax)
ax.set_title('Sale Price Distribution by Borough')
ax.set_xlabel('Borough')
ax.set_ylabel('Sale Price ($)')
plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.savefig('../notebooks/price_by_borough.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── TEST 3: T-TEST - RESIDENTIAL VS NON-RESIDENTIAL PROPERTIES ──────────
if 'is_residential' in analysis_df.columns:
    print("\n" + "="*80)
    print("INDEPENDENT T-TEST - Residential vs Non-Residential")
    print("="*80)
    print("Null Hypothesis: Mean sale price is equal for residential and non-residential")
    
    residential_prices = analysis_df[analysis_df['is_residential'] == 1]['sale_price'].dropna()
    non_residential_prices = analysis_df[analysis_df['is_residential'] == 0]['sale_price'].dropna()
    
    t_stat, p_value = ttest_ind(residential_prices, non_residential_prices)
    
    print(f"\nResidential (n={len(residential_prices)}):")
    print(f"  Mean: ${residential_prices.mean():,.0f}")
    print(f"  Std: ${residential_prices.std():,.0f}")
    
    print(f"\nNon-Residential (n={len(non_residential_prices)}):")
    print(f"  Mean: ${non_residential_prices.mean():,.0f}")
    print(f"  Std: ${non_residential_prices.std():,.0f}")
    
    print(f"\nT-Statistic: {t_stat:.6f}")
    print(f"p-value: {p_value:.6e}")
    print(f"\nConclusion: {'Significant difference' if p_value < 0.05 else 'No significant difference'} (α=0.05)")

In [ ]:
# ── TEST 4: CHI-SQUARE TEST - BUILDING CLASS & PRICE BRACKET ────────────
if 'building_class_description' in analysis_df.columns and 'price_bracket' in analysis_df.columns:
    print("\n" + "="*80)
    print("CHI-SQUARE TEST - Building Class vs Price Bracket")
    print("="*80)
    print("Null Hypothesis: Building class and price bracket are independent")
    
    # Create contingency table (limiting to top classes for clarity)
    top_classes = analysis_df['building_class_description'].value_counts().head(5).index
    contingency_table = pd.crosstab(
        analysis_df[analysis_df['building_class_description'].isin(top_classes)]['building_class_description'],
        analysis_df[analysis_df['building_class_description'].isin(top_classes)]['price_bracket']
    )
    
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    print(f"\nChi-Square Statistic: {chi2:.6f}")
    print(f"p-value: {p_value:.6e}")
    print(f"Degrees of Freedom: {dof}")
    print(f"\nConclusion: {'Significant association' if p_value < 0.05 else 'No significant association'} (α=0.05)")
    
    print("\nContingency Table (Top 5 Building Classes):")
    print(contingency_table)

## 5. Regression Analysis

In [ ]:
# ── SIMPLE LINEAR REGRESSION: GROSS_SQUARE_FEET → SALE_PRICE ────────────
print("\n" + "="*80)
print("SIMPLE LINEAR REGRESSION")
print("Model: Sale Price ~ Gross Square Feet")
print("="*80)

# Prepare data
reg_data = analysis_df[['gross_square_feet', 'sale_price']].dropna()
X_simple = reg_data[['gross_square_feet']].values
y = reg_data['sale_price'].values

# Fit model
model_simple = LinearRegression()
model_simple.fit(X_simple, y)

# Predictions & metrics
y_pred_simple = model_simple.predict(X_simple)
r2_simple = r2_score(y, y_pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y, y_pred_simple))

print(f"\nCoefficients:")
print(f"  Intercept: ${model_simple.intercept_:,.2f}")
print(f"  Slope (Gross Sq Ft): ${model_simple.coef_[0]:,.2f}")
print(f"\nModel Performance:")
print(f"  R-squared: {r2_simple:.4f}")
print(f"  RMSE: ${rmse_simple:,.0f}")
print(f"\nInterpretation: Each additional square foot increases sale price by ${model_simple.coef_[0]:,.2f}")

In [ ]:
# ── VISUALIZE SIMPLE LINEAR REGRESSION ──────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
# Sample for clarity (plot every 10th point)
sample_indices = np.random.choice(len(reg_data), size=min(2000, len(reg_data)), replace=False)
ax.scatter(X_simple[sample_indices], y[sample_indices], alpha=0.3, s=20, label='Actual Data')
ax.plot(X_simple[:, 0], y_pred_simple, color='red', linewidth=2, label='Regression Line')
ax.set_xlabel('Gross Square Feet')
ax.set_ylabel('Sale Price ($)')
ax.set_title(f'Linear Regression: Sale Price vs Gross Square Feet (R²={r2_simple:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/simple_regression.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── RESIDUAL ANALYSIS FOR SIMPLE REGRESSION ────────────────────────────
residuals = y - y_pred_simple

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Fitted
axes[0, 0].scatter(y_pred_simple[sample_indices], residuals[sample_indices], alpha=0.3, s=20)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted Values')
axes[0, 0].grid(True, alpha=0.3)

# Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')
axes[0, 1].grid(True, alpha=0.3)

# Histogram of residuals
axes[1, 0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')
axes[1, 0].grid(True, alpha=0.3)

# Residuals over order
axes[1, 1].plot(residuals, alpha=0.5)
axes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Observation Order')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Residuals vs Order')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../notebooks/residual_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── MULTIPLE LINEAR REGRESSION ──────────────────────────────────────────
print("\n" + "="*80)
print("MULTIPLE LINEAR REGRESSION")
print("="*80)

# Select relevant features
feature_cols = ['gross_square_feet', 'building_age', 'residential_units', 'commercial_units']
feature_cols = [col for col in feature_cols if col in analysis_df.columns]

# Prepare data
multi_reg_data = analysis_df[feature_cols + ['sale_price']].dropna()
X_multi = multi_reg_data[feature_cols].values
y_multi = multi_reg_data['sale_price'].values

# Fit model
model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

# Predictions & metrics
y_pred_multi = model_multi.predict(X_multi)
r2_multi = r2_score(y_multi, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_multi, y_pred_multi))

# Adjusted R-squared
n = len(multi_reg_data)
p = len(feature_cols)
adj_r2_multi = 1 - (1 - r2_multi) * (n - 1) / (n - p - 1)

print(f"\nModel Formula: Sale Price ~ {' + '.join(feature_cols)}")
print(f"\nCoefficients:")
print(f"  Intercept: ${model_multi.intercept_:,.2f}")
for feature, coef in zip(feature_cols, model_multi.coef_):
    print(f"  {feature}: {coef:,.4f}")

print(f"\nModel Performance:")
print(f"  R-squared: {r2_multi:.4f}")
print(f"  Adjusted R-squared: {adj_r2_multi:.4f}")
print(f"  RMSE: ${rmse_multi:,.0f}")
print(f"  Samples: {n}")

In [ ]:
# ── FEATURE IMPORTANCE (COEFFICIENT MAGNITUDES) ────────────────────────
# Standardize coefficients for comparison
scaler = StandardScaler()
X_multi_scaled = scaler.fit_transform(X_multi)
model_multi_scaled = LinearRegression()
model_multi_scaled.fit(X_multi_scaled, y_multi)

coef_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model_multi.coef_,
    'Standardized_Coefficient': model_multi_scaled.coef_,
    'Abs_Std_Coefficient': np.abs(model_multi_scaled.coef_)
}).sort_values('Abs_Std_Coefficient', ascending=False)

print("\nFeature Importance (Standardized Coefficients):")
print(coef_importance)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
coef_importance.sort_values('Standardized_Coefficient').plot(x='Feature', y='Standardized_Coefficient', 
                                                              kind='barh', ax=ax, color='steelblue', legend=False)
ax.set_title('Feature Importance in Multiple Regression Model')
ax.set_xlabel('Standardized Coefficient')
plt.tight_layout()
plt.savefig('../notebooks/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── ACTUAL VS PREDICTED (MULTIPLE REGRESSION) ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
sample_idx = np.random.choice(len(multi_reg_data), size=min(1000, len(multi_reg_data)), replace=False)
ax.scatter(y_multi[sample_idx], y_pred_multi[sample_idx], alpha=0.3, s=20)
# Perfect prediction line
min_val = min(y_multi.min(), y_pred_multi.min())
max_val = max(y_multi.max(), y_pred_multi.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Sale Price ($)')
ax.set_ylabel('Predicted Sale Price ($)')
ax.set_title(f'Multiple Regression: Actual vs Predicted (R²={r2_multi:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../notebooks/actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Key Findings & Summary

In [ ]:
# ── SUMMARY OF KEY STATISTICAL FINDINGS ────────────────────────────────
print("="*80)
print("STATISTICAL ANALYSIS SUMMARY")
print("="*80)

print("\n1. CORRELATION FINDINGS:")
print(f"   - Strongest predictor of sale price (Pearson): {price_corr.index[1]} (r={price_corr.iloc[1]:.4f})")
print(f"   - Second strongest: {price_corr.index[2]} (r={price_corr.iloc[2]:.4f})")

print("\n2. HYPOTHESIS TEST RESULTS:")
print(f"   - Normality (Shapiro-Wilk): Most variables show non-normal distributions")
print(f"   - ANOVA (Borough Effect): {'Significant' if p_value < 0.05 else 'Not significant'} differences in mean prices across boroughs")

print("\n3. REGRESSION MODELS:")
print(f"   - Simple Model (Gross Sq Ft only): R² = {r2_simple:.4f}")
print(f"   - Multiple Model ({len(feature_cols)} features): R² = {r2_multi:.4f}, Adjusted R² = {adj_r2_multi:.4f}")
print(f"   - Model Improvement: {(r2_multi - r2_simple)*100:.2f}% increase in R²")

print("\n4. RESIDUAL DIAGNOSTICS:")
print(f"   - Mean Residual: ${residuals.mean():,.2f} (should be ~0)")
print(f"   - Std Dev Residuals: ${residuals.std():,.0f}")

print("\n" + "="*80)

In [ ]:
# ── EXPORT KEY RESULTS TO CSV FOR TABLEAU/REPORTING ────────────────────
# Correlation summary
corr_export = price_corr[1:].reset_index()
corr_export.columns = ['Variable', 'Correlation_with_Price']
corr_export.to_csv('../data/processed/correlation_summary.csv', index=False)

# Borough price summary
borough_price_stats.to_csv('../data/processed/borough_price_summary.csv')

# Model results
model_results = pd.DataFrame({
    'Model': ['Simple (Gross Sq Ft)', 'Multiple (4 Features)'],
    'R_Squared': [r2_simple, r2_multi],
    'Adjusted_R_Squared': [np.nan, adj_r2_multi],
    'RMSE': [rmse_simple, rmse_multi]
})
model_results.to_csv('../data/processed/model_performance.csv', index=False)

print("\nExport Summary:")
print("✓ Correlation summary saved to: data/processed/correlation_summary.csv")
print("✓ Borough price summary saved to: data/processed/borough_price_summary.csv")
print("✓ Model performance saved to: data/processed/model_performance.csv")
print("\nVisualization files saved to notebooks/:")
print("  - stat_distributions.png")
print("  - correlation_heatmap.png")
print("  - price_correlations.png")
print("  - price_by_borough.png")
print("  - simple_regression.png")
print("  - residual_analysis.png")
print("  - feature_importance.png")
print("  - actual_vs_predicted.png")